In [1]:
# Cell 1: Install required packages
!pip install openai chromadb tiktoken --quiet
print("✅ Packages installed successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.2 which is incompatible.
opentelemetry-exporter-prometheus 0.56b0 requires opentelemetry-sdk~=1.35.0, but you have opentelemetry-sdk 1.39.0 which is incompatible.
opencensus-ext-azure 1.1.14 requires psutil>=5.6.3, but you have psutil 5.2.2 which is incompatible.
mlflow-skinny 2.21.3 requires packaging<25, but you have packaging 25.0 which is incompatible.
mlflow-skinny 2.21.3 requires protobuf<6,>=3.12.0, but you have protobuf 6.33.2 which is incompatible.
distributed 2023.2.0 requires psutil>=5.7.0, but you have psutil 5.2.2 which is incompatible.
dask-sql 2024.5.0 requires dask[dat

In [2]:
# Cell 2: Configure Azure OpenAI credentials
import os

# Embedding model (East US)
EMBEDDING_ENDPOINT = "https://rag-demo-openai-najib.openai.azure.com/"
EMBEDDING_KEY = "22eglYDwbQGJhdtByt2SeLDNVS1i5PnAW0R3nqnq4AGX3ml3tNi3JQQJ99BLACYeBjFX"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"

# GPT model (East US 2)
GPT_ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"
GPT_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
GPT_DEPLOYMENT = "gpt-4o-mini"

print("✅ Credentials configured!")
print(f"   Embedding endpoint: {EMBEDDING_ENDPOINT}")
print(f"   GPT endpoint: {GPT_ENDPOINT}")

✅ Credentials configured!
   Embedding endpoint: https://rag-demo-openai-najib.openai.azure.com/
   GPT endpoint: https://najr-miyonro1-eastus2.cognitiveservices.azure.com/


In [3]:
# Cell 3: Ericsson Knowledge Base - Content for RAG Demo
# This represents internal documentation about Ericsson's 5G and AI capabilities

ERICSSON_KNOWLEDGE_BASE = """
# Ericsson 5G Network Solutions

## Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases such as enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC). Network slicing allows operators to offer differentiated services to enterprise customers, including dedicated bandwidth, guaranteed latency, and custom security policies. Ericsson's slice management system provides end-to-end orchestration across radio, transport, and core networks.

## Ericsson Spectrum Sharing
Ericsson Spectrum Sharing (ESS) allows operators to dynamically share spectrum between 4G LTE and 5G NR on the same frequency band. This technology uses intelligent algorithms to allocate spectrum resources in real-time based on traffic demand. ESS enables faster 5G deployment without requiring dedicated spectrum, reducing time-to-market from years to months. The solution supports both FDD and TDD spectrum bands and can achieve up to 50% faster 5G coverage rollout.

## Cloud RAN Architecture
Ericsson's Cloud RAN solution virtualizes radio access network functions, enabling deployment on commercial off-the-shelf (COTS) hardware. The architecture separates the centralized unit (CU), distributed unit (DU), and radio unit (RU) for flexible deployment options. Cloud RAN reduces total cost of ownership by up to 40% through hardware consolidation and simplified operations. It supports Open RAN interfaces for multi-vendor interoperability while maintaining carrier-grade performance.

## AI-Powered Network Operations
Ericsson's AI-driven operations platform uses machine learning to predict network issues before they impact subscribers. The system analyzes millions of data points from network elements to identify patterns and anomalies. Predictive maintenance algorithms can forecast equipment failures with 95% accuracy up to 7 days in advance. AI-based traffic optimization automatically adjusts network parameters to maintain optimal performance during peak usage.

## Energy Efficiency Solutions
Ericsson's energy-efficient 5G products reduce power consumption by up to 25% compared to previous generations. Advanced sleep modes automatically power down components during low-traffic periods without affecting service quality. The company's Breaking the Energy Curve initiative aims to achieve zero net carbon emissions by 2040. Site solutions include solar panels, battery storage, and intelligent power management systems.

## Ericsson Private Networks
Ericsson Private 5G Networks provide dedicated connectivity for enterprise and industrial applications. These networks offer complete isolation from public networks with local data processing for enhanced security. Use cases include smart manufacturing with automated guided vehicles, port operations with remote-controlled cranes, and mining operations with autonomous vehicles. Private networks can be deployed on-premises or as a managed service with guaranteed SLAs.

## Edge Computing Integration
Ericsson's edge computing solutions bring processing power closer to users, reducing latency to under 10 milliseconds. The Ericsson Edge Gravity platform enables application developers to deploy services at the network edge. Integration with major cloud providers including AWS, Azure, and Google Cloud allows seamless hybrid deployments. Edge computing is essential for applications like augmented reality, autonomous vehicles, and real-time video analytics.

## Security Framework
Ericsson's 5G security framework implements zero-trust architecture with continuous verification of all network entities. The solution includes hardware-based security anchors, encrypted communications, and real-time threat detection. Compliance with 3GPP security standards ensures interoperability with other vendors' equipment. Security operations centers provide 24/7 monitoring and incident response capabilities.
"""

print("✅ Ericsson Knowledge Base created!")
print(f"   Total characters: {len(ERICSSON_KNOWLEDGE_BASE):,}")
print(f"   Approximate words: {len(ERICSSON_KNOWLEDGE_BASE.split()):,}")

✅ Ericsson Knowledge Base created!
   Total characters: 4,063
   Approximate words: 527


In [4]:
# Cell 4: Demonstrate Different Chunking Strategies
# This shows how the same text can be split differently - WITH EXAMPLES

# === Strategy 1: Fixed-Size Chunking ===
def fixed_size_chunking(text, chunk_size=500, overlap=50):
    """Split text into fixed-size chunks with overlap"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if c]

# === Strategy 2: Sentence-Based Chunking ===
def sentence_chunking(text, sentences_per_chunk=3):
    """Split text by sentences, grouping N sentences per chunk"""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i+sentences_per_chunk])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

# === Strategy 3: Section-Based Chunking ===
def section_chunking(text):
    """Split text by sections (## headers)"""
    import re
    sections = re.split(r'\n##\s+', text)
    chunks = []
    for section in sections:
        if section.strip() and len(section.strip()) > 50:
            chunks.append(section.strip())
    return chunks

# Apply all three strategies
fixed_chunks = fixed_size_chunking(ERICSSON_KNOWLEDGE_BASE, chunk_size=500, overlap=50)
sentence_chunks = sentence_chunking(ERICSSON_KNOWLEDGE_BASE, sentences_per_chunk=3)
section_chunks = section_chunking(ERICSSON_KNOWLEDGE_BASE)

# === SHOW THE DIFFERENCE ===
print("=" * 70)
print("🔍 CHUNKING STRATEGY COMPARISON - SEE THE DIFFERENCE!")
print("=" * 70)

# Strategy 1: Fixed-Size
print("\n" + "━" * 70)
print("📊 STRATEGY 1: FIXED-SIZE CHUNKING (500 chars)")
print("━" * 70)
print(f"Total chunks: {len(fixed_chunks)}")
print("\n🔹 Chunk 1 (first 500 chars):")
print("-" * 40)
print(fixed_chunks[0][:500])
print("-" * 40)
print("\n⚠️  PROBLEM: See how it cuts mid-sentence? This loses context!")
print(f"   Chunk ends with: '...{fixed_chunks[0][-50:]}'")

# Strategy 2: Sentence-Based  
print("\n" + "━" * 70)
print("📊 STRATEGY 2: SENTENCE-BASED CHUNKING (3 sentences)")
print("━" * 70)
print(f"Total chunks: {len(sentence_chunks)}")
print("\n🔹 Chunk 1:")
print("-" * 40)
print(sentence_chunks[0])
print("-" * 40)
print("\n✅ BETTER: Complete sentences preserved!")

# Strategy 3: Section-Based
print("\n" + "━" * 70)
print("📊 STRATEGY 3: SECTION-BASED CHUNKING (by topic)")
print("━" * 70)
print(f"Total chunks: {len(section_chunks)}")
print("\n🔹 Chunk 1 (Network Slicing section):")
print("-" * 40)
print(section_chunks[0][:400] + "...")
print("-" * 40)
print("\n✅ BEST: Each chunk is a complete topic - ideal for RAG retrieval!")

# Summary Table
print("\n" + "=" * 70)
print("📋 SUMMARY: Which Strategy to Use?")
print("=" * 70)
print("""
| Strategy      | Chunks | Avg Size | Pros                    | Cons                    |
|---------------|--------|----------|-------------------------|-------------------------|
| Fixed-Size    |   {}    | {} chars | Simple, predictable     | Cuts mid-sentence       |
| Sentence      |   {}    | {} chars | Preserves sentences     | May split related info  |
| Section       |   {}    | {} chars | Topic-coherent chunks   | Uneven chunk sizes      |
""".format(
    len(fixed_chunks), sum(len(c) for c in fixed_chunks)//len(fixed_chunks),
    len(sentence_chunks), sum(len(c) for c in sentence_chunks)//len(sentence_chunks),
    len(section_chunks), sum(len(c) for c in section_chunks)//len(section_chunks)
))

print("💡 For RAG: Section/Semantic chunking usually gives best retrieval results!")

🔍 CHUNKING STRATEGY COMPARISON - SEE THE DIFFERENCE!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 STRATEGY 1: FIXED-SIZE CHUNKING (500 chars)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total chunks: 10

🔹 Chunk 1 (first 500 chars):
----------------------------------------
# Ericsson 5G Network Solutions

## Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases such as enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC). Network slicing allows operators to offer differentiated services to enterprise customers, including dedicated bandwidth, gua
----------------------------------------

⚠️  PROBLEM: See how it cuts mid-sentence? This loses context!
   Chunk ends with: '...rise customers, including dedicated bandw

In [5]:
# Cell 5: Understanding Semantic Chunking
print("=" * 70)
print("🧠 WHAT IS SEMANTIC CHUNKING?")
print("=" * 70)

print("""
CHUNKING HIERARCHY (Simple → Advanced):

1. FIXED-SIZE CHUNKING
   ├── Splits every N characters
   ├── Fast and simple
   └── ❌ Problem: Cuts mid-sentence, loses context

2. SENTENCE-BASED CHUNKING  
   ├── Splits at sentence boundaries (. ! ?)
   ├── Preserves grammatical units
   └── ⚠️ Problem: May split related sentences apart

3. SECTION/PARAGRAPH CHUNKING (what we did)
   ├── Splits at headers, paragraphs, or logical breaks
   ├── Keeps related content together
   └── ✅ Good for structured documents

4. TRUE SEMANTIC CHUNKING (most advanced)
   ├── Uses AI embeddings to measure similarity between sentences
   ├── Detects where the MEANING changes significantly
   ├── Creates chunks based on topic coherence
   └── ✅ Best retrieval quality, but more compute cost

""")

print("=" * 70)
print("💡 FOR YOUR ERICSSON PROJECT:")
print("=" * 70)
print("""
- Our SECTION-BASED approach is a practical form of semantic chunking
- Each chunk = one Ericsson topic (Network Slicing, Cloud RAN, etc.)
- This works well because the source document is already well-structured

- For unstructured documents (emails, transcripts, PDFs), use:
  - LangChain's SemanticChunker
  - LlamaIndex's SemanticSplitter
  - These use embeddings to find natural topic boundaries
""")

# Show our section chunks as "semantic" chunks for the demo
print("\n" + "=" * 70)
print("📦 OUR SEMANTIC CHUNKS (Section-Based):")
print("=" * 70)
for i, chunk in enumerate(section_chunks[:4], 1):
    # Extract topic name from chunk
    topic = chunk.split('\n')[0][:50]
    print(f"\n  Chunk {i}: {topic}...")
    print(f"           ({len(chunk)} chars)")

🧠 WHAT IS SEMANTIC CHUNKING?

CHUNKING HIERARCHY (Simple → Advanced):

1. FIXED-SIZE CHUNKING
   ├── Splits every N characters
   ├── Fast and simple
   └── ❌ Problem: Cuts mid-sentence, loses context

2. SENTENCE-BASED CHUNKING  
   ├── Splits at sentence boundaries (. ! ?)
   ├── Preserves grammatical units
   └── ⚠️ Problem: May split related sentences apart

3. SECTION/PARAGRAPH CHUNKING (what we did)
   ├── Splits at headers, paragraphs, or logical breaks
   ├── Keeps related content together
   └── ✅ Good for structured documents

4. TRUE SEMANTIC CHUNKING (most advanced)
   ├── Uses AI embeddings to measure similarity between sentences
   ├── Detects where the MEANING changes significantly
   ├── Creates chunks based on topic coherence
   └── ✅ Best retrieval quality, but more compute cost


💡 FOR YOUR ERICSSON PROJECT:

- Our SECTION-BASED approach is a practical form of semantic chunking
- Each chunk = one Ericsson topic (Network Slicing, Cloud RAN, etc.)
- This works well bec

In [6]:
# Cell 6: Understanding Embeddings - Models and Comparison
print("=" * 70)
print("🧮 UNDERSTANDING EMBEDDINGS")
print("=" * 70)

print("""
WHAT IS AN EMBEDDING?
━━━━━━━━━━━━━━━━━━━━
An embedding converts text into a list of numbers (vector) that captures 
the MEANING of the text. Similar meanings = similar vectors.

Example:
  "Ericsson 5G network" → [0.023, -0.041, 0.089, ..., 0.012]  (1536 numbers)
  "5G mobile technology" → [0.021, -0.038, 0.092, ..., 0.015]  (similar!)
  "Italian pizza recipe" → [-0.056, 0.078, -0.012, ..., 0.067] (very different)

""")

print("=" * 70)
print("📊 EMBEDDING MODELS COMPARISON")
print("=" * 70)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                        EMBEDDING MODELS COMPARISON                          │
├──────────────────────┬───────────┬──────────┬───────────────────────────────┤
│ Model                │ Dimensions│ Cost     │ Best For                      │
├──────────────────────┼───────────┼──────────┼───────────────────────────────┤
│ text-embedding-ada-002│   1536   │ $0.0001/1K│ General purpose (we use this)│
│ text-embedding-3-small│   1536   │ $0.00002/1K│ Cost-effective, good quality │
│ text-embedding-3-large│   3072   │ $0.00013/1K│ Highest accuracy             │
├──────────────────────┼───────────┼──────────┼───────────────────────────────┤
│ Amazon Titan         │   1536   │ AWS pricing│ AWS Bedrock environments     │
│ Cohere embed-v3      │   1024   │ $0.0001/1K│ Multilingual support         │
├──────────────────────┼───────────┼──────────┼───────────────────────────────┤
│ all-MiniLM-L6-v2     │    384   │ FREE     │ Local/offline (open source)   │
│ BGE-large-en         │   1024   │ FREE     │ High quality open source      │
│ E5-large-v2          │   1024   │ FREE     │ Best open source quality      │
└──────────────────────┴───────────┴──────────┴───────────────────────────────┘

KEY INSIGHTS:
- Higher dimensions = more semantic detail, but more storage/compute
- OpenAI models are easy to use but have API costs
- Open source models (MiniLM, BGE, E5) are FREE but need local GPU
""")

print("\n" + "=" * 70)
print("🔍 HOW SIMILARITY SEARCH WORKS")
print("=" * 70)

print("""
STEP 1: Index Time (One-time)
┌──────────────────┐      ┌─────────────┐      ┌─────────────────┐
│ "Network Slicing │ ──▶  │  Embedding  │ ──▶  │ [0.02, -0.04,   │
│  enables..."     │      │    Model    │      │  0.08, ...]     │
└──────────────────┘      └─────────────┘      └────────┬────────┘
                                                        │
                                                        ▼
                                               ┌─────────────────┐
                                               │  Vector Store   │
                                               │   (ChromaDB)    │
                                               └─────────────────┘

STEP 2: Query Time (Every search)
┌──────────────────┐      ┌─────────────┐      ┌─────────────────┐
│ "What is network │ ──▶  │  Embedding  │ ──▶  │ [0.03, -0.03,   │
│  slicing?"       │      │    Model    │      │  0.09, ...]     │
└──────────────────┘      └─────────────┘      └────────┬────────┘
                                                        │
                                         ┌──────────────┘
                                         ▼
                          ┌─────────────────────────────────────┐
                          │   Compare with all stored vectors   │
                          │   using COSINE SIMILARITY           │
                          │                                     │
                          │   Query ● ─────── 0.95 similarity   │
                          │              ╲                      │
                          │               ╲─── 0.72 similarity  │
                          │                ╲                    │
                          │                 ╲── 0.31 similarity │
                          └─────────────────────────────────────┘
                                         │
                                         ▼
                          Return top K most similar chunks!
""")

print("\n" + "=" * 70)
print("📏 SIMILARITY METRICS")
print("=" * 70)

print("""
Three ways to measure similarity between vectors:

1. COSINE SIMILARITY (Most Common for Text)
   ├── Measures the angle between vectors
   ├── Range: -1 to 1 (1 = identical meaning)
   ├── Ignores magnitude, focuses on direction
   └── ✅ Best for: Text similarity, RAG

2. EUCLIDEAN DISTANCE (L2)
   ├── Measures straight-line distance
   ├── Range: 0 to ∞ (0 = identical)
   ├── Sensitive to vector magnitude
   └── ✅ Best for: Image similarity, clustering

3. DOT PRODUCT
   ├── Simple multiplication of vectors
   ├── Range: -∞ to ∞ (higher = more similar)
   ├── Fast to compute
   └── ✅ Best for: When vectors are normalized

For RAG: Use COSINE SIMILARITY (default in most vector DBs)
""")

🧮 UNDERSTANDING EMBEDDINGS

WHAT IS AN EMBEDDING?
━━━━━━━━━━━━━━━━━━━━
An embedding converts text into a list of numbers (vector) that captures 
the MEANING of the text. Similar meanings = similar vectors.

Example:
  "Ericsson 5G network" → [0.023, -0.041, 0.089, ..., 0.012]  (1536 numbers)
  "5G mobile technology" → [0.021, -0.038, 0.092, ..., 0.015]  (similar!)
  "Italian pizza recipe" → [-0.056, 0.078, -0.012, ..., 0.067] (very different)


📊 EMBEDDING MODELS COMPARISON

┌─────────────────────────────────────────────────────────────────────────────┐
│                        EMBEDDING MODELS COMPARISON                          │
├──────────────────────┬───────────┬──────────┬───────────────────────────────┤
│ Model                │ Dimensions│ Cost     │ Best For                      │
├──────────────────────┼───────────┼──────────┼───────────────────────────────┤
│ text-embedding-ada-002│   1536   │ $0.0001/1K│ General purpose (we use this)│
│ text-embedding-3-small│   1536   │ $0.

In [7]:
# Cell 7: Create Embeddings and Store in Vector Database
from openai import AzureOpenAI
import chromadb
import numpy as np

print("=" * 70)
print("🚀 CREATING EMBEDDINGS AND VECTOR STORE")
print("=" * 70)

# Initialize Azure OpenAI client for embeddings
embedding_client = AzureOpenAI(
    api_key=EMBEDDING_KEY,
    api_version="2024-02-01",
    azure_endpoint=EMBEDDING_ENDPOINT
)

# Function to get embeddings
def get_embedding(text):
    """Convert text to vector using Azure OpenAI"""
    response = embedding_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding

# Step 1: Test embedding
print("\n📍 Step 1: Testing Embedding API...")
test_text = "Ericsson 5G network slicing"
test_embedding = get_embedding(test_text)
print(f"   ✅ Success! Text: '{test_text}'")
print(f"   ✅ Vector dimension: {len(test_embedding)}")
print(f"   ✅ First 5 values: {test_embedding[:5]}")

# Step 2: Create ChromaDB vector store
print("\n📍 Step 2: Creating Vector Database (ChromaDB)...")
chroma_client = chromadb.Client()

# Delete collection if exists (for re-runs)
try:
    chroma_client.delete_collection("ericsson_knowledge")
except:
    pass

collection = chroma_client.create_collection(
    name="ericsson_knowledge",
    metadata={"description": "Ericsson 5G and AI documentation"}
)
print("   ✅ Collection 'ericsson_knowledge' created!")

# Step 3: Embed and store each chunk
print("\n📍 Step 3: Embedding and Storing Chunks...")
print("-" * 50)

chunk_embeddings = []  # Store for later visualization

for i, chunk in enumerate(section_chunks):
    # Get embedding for this chunk
    embedding = get_embedding(chunk)
    chunk_embeddings.append(embedding)
    
    # Extract topic name for display
    topic = chunk.split('\n')[0][:40]
    
    # Add to vector store
    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[chunk],
        metadatas=[{"chunk_id": i, "topic": topic}]
    )
    
    print(f"   ✅ Chunk {i+1}: '{topic}...'")
    print(f"      Vector: [{embedding[0]:.4f}, {embedding[1]:.4f}, ... {embedding[-1]:.4f}]")

# Summary
print("\n" + "=" * 70)
print("✅ VECTOR DATABASE READY!")
print("=" * 70)
print(f"""
   📊 Total chunks stored: {collection.count()}
   📊 Vector dimensions: {len(test_embedding)}
   📊 Database: ChromaDB (in-memory)
   
   Each Ericsson topic is now a {len(test_embedding)}-dimensional vector!
   We can now search by MEANING, not just keywords.
""")

ModuleNotFoundError: No module named 'openai'

In [8]:
# Cell 8: Install openai package properly
!pip install openai --quiet
print("✅ OpenAI package installed!")

✅ OpenAI package installed!


In [9]:
# Cell 9: Create Embeddings and Store in Vector Database
from openai import AzureOpenAI
import chromadb

print("=" * 70)
print("🚀 CREATING EMBEDDINGS AND VECTOR STORE")
print("=" * 70)

# Initialize Azure OpenAI client for embeddings
embedding_client = AzureOpenAI(
    api_key=EMBEDDING_KEY,
    api_version="2024-02-01",
    azure_endpoint=EMBEDDING_ENDPOINT
)

# Function to get embeddings
def get_embedding(text):
    """Convert text to vector using Azure OpenAI"""
    response = embedding_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding

# Step 1: Test embedding
print("\n📍 Step 1: Testing Embedding API...")
test_text = "Ericsson 5G network slicing"
test_embedding = get_embedding(test_text)
print(f"   ✅ Success! Text: '{test_text}'")
print(f"   ✅ Vector dimension: {len(test_embedding)}")
print(f"   ✅ First 5 values: {test_embedding[:5]}")

# Step 2: Create ChromaDB vector store
print("\n📍 Step 2: Creating Vector Database (ChromaDB)...")
chroma_client = chromadb.Client()

# Delete collection if exists (for re-runs)
try:
    chroma_client.delete_collection("ericsson_knowledge")
except:
    pass

collection = chroma_client.create_collection(
    name="ericsson_knowledge",
    metadata={"description": "Ericsson 5G and AI documentation"}
)
print("   ✅ Collection 'ericsson_knowledge' created!")

# Step 3: Embed and store each chunk
print("\n📍 Step 3: Embedding and Storing Chunks...")
print("-" * 50)

for i, chunk in enumerate(section_chunks):
    embedding = get_embedding(chunk)
    topic = chunk.split('\n')[0][:40]
    
    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[chunk],
        metadatas=[{"chunk_id": i, "topic": topic}]
    )
    
    print(f"   ✅ Chunk {i+1}: '{topic}...'")

# Summary
print("\n" + "=" * 70)
print("✅ VECTOR DATABASE READY!")
print("=" * 70)
print(f"   📊 Total chunks stored: {collection.count()}")
print(f"   📊 Vector dimensions: {len(test_embedding)}")

ModuleNotFoundError: No module named 'openai'

In [1]:
# Cell: Complete Setup - Run this after kernel restart
# Install packages
!pip install openai chromadb --quiet

# Configuration
import os
EMBEDDING_ENDPOINT = "https://rag-demo-openai-najib.openai.azure.com/"
EMBEDDING_KEY = "22eglYDwbQGJhdtByt2SeLDNVS1i5PnAW0R3nqnq4AGX3ml3tNi3JQQJ99BLACYeBjFX"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"
GPT_ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"
GPT_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
GPT_DEPLOYMENT = "gpt-4o-mini"

# Ericsson Knowledge Base
ERICSSON_KNOWLEDGE_BASE = """
## Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases such as enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC). Network slicing allows operators to offer differentiated services to enterprise customers.

## Ericsson Spectrum Sharing
Ericsson Spectrum Sharing (ESS) allows operators to dynamically share spectrum between 4G LTE and 5G NR on the same frequency band. This technology uses intelligent algorithms to allocate spectrum resources in real-time based on traffic demand. ESS enables faster 5G deployment without requiring dedicated spectrum.

## Cloud RAN Architecture
Ericsson's Cloud RAN solution virtualizes radio access network functions, enabling deployment on commercial off-the-shelf hardware. The architecture separates the centralized unit (CU), distributed unit (DU), and radio unit (RU) for flexible deployment. Cloud RAN reduces total cost of ownership by up to 40%.

## AI-Powered Network Operations
Ericsson's AI-driven operations platform uses machine learning to predict network issues before they impact subscribers. The system analyzes millions of data points to identify patterns and anomalies. Predictive maintenance algorithms can forecast equipment failures with 95% accuracy.

## Energy Efficiency Solutions
Ericsson's energy-efficient 5G products reduce power consumption by up to 25% compared to previous generations. Advanced sleep modes automatically power down components during low-traffic periods. The company aims to achieve zero net carbon emissions by 2040.

## Private Networks
Ericsson Private 5G Networks provide dedicated connectivity for enterprise and industrial applications. Use cases include smart manufacturing, port operations, and mining with autonomous vehicles. Private networks can be deployed on-premises with guaranteed SLAs.

## Edge Computing
Ericsson's edge computing solutions bring processing power closer to users, reducing latency to under 10 milliseconds. Integration with AWS, Azure, and Google Cloud allows seamless hybrid deployments. Edge computing enables augmented reality and autonomous vehicles.

## Security Framework
Ericsson's 5G security framework implements zero-trust architecture with continuous verification. The solution includes hardware-based security anchors and encrypted communications. Security operations centers provide 24/7 monitoring.
"""

# Create section chunks
import re
sections = re.split(r'\n##\s+', ERICSSON_KNOWLEDGE_BASE)
section_chunks = [s.strip() for s in sections if s.strip() and len(s.strip()) > 50]

print("✅ Configuration loaded!")
print(f"✅ Knowledge base: {len(section_chunks)} chunks created")

# Now create embeddings
from openai import AzureOpenAI
import chromadb

embedding_client = AzureOpenAI(
    api_key=EMBEDDING_KEY,
    api_version="2024-02-01",
    azure_endpoint=EMBEDDING_ENDPOINT
)

def get_embedding(text):
    response = embedding_client.embeddings.create(input=text, model=EMBEDDING_DEPLOYMENT)
    return response.data[0].embedding

# Create vector store
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("ericsson_knowledge")
except:
    pass

collection = chroma_client.create_collection(name="ericsson_knowledge")

print("\n🔄 Creating embeddings...")
for i, chunk in enumerate(section_chunks):
    embedding = get_embedding(chunk)
    topic = chunk.split('\n')[0][:30]
    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[chunk],
        metadatas=[{"chunk_id": i, "topic": topic}]
    )
    print(f"   ✅ Chunk {i+1}: {topic}...")

print(f"\n✅ Vector database ready with {collection.count()} chunks!")

✅ Configuration loaded!
✅ Knowledge base: 8 chunks created


ModuleNotFoundError: No module named 'openai'

In [2]:
!pip install openai chromadb --quiet
print("✅ Packages installed! Now run the next cell.")

✅ Packages installed! Now run the next cell.


In [3]:
# Cell 2: Setup and Create Embeddings
from openai import AzureOpenAI
import chromadb
import re

# Configuration
EMBEDDING_ENDPOINT = "https://rag-demo-openai-najib.openai.azure.com/"
EMBEDDING_KEY = "22eglYDwbQGJhdtByt2SeLDNVS1i5PnAW0R3nqnq4AGX3ml3tNi3JQQJ99BLACYeBjFX"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"
GPT_ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"
GPT_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
GPT_DEPLOYMENT = "gpt-4o-mini"

# Ericsson Knowledge Base
ERICSSON_KB = """
## Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases such as enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC).

## Ericsson Spectrum Sharing
Ericsson Spectrum Sharing (ESS) allows operators to dynamically share spectrum between 4G LTE and 5G NR on the same frequency band. This technology uses intelligent algorithms to allocate spectrum resources in real-time based on traffic demand.

## Cloud RAN Architecture
Ericsson's Cloud RAN solution virtualizes radio access network functions, enabling deployment on commercial off-the-shelf hardware. Cloud RAN reduces total cost of ownership by up to 40% through hardware consolidation.

## AI-Powered Network Operations
Ericsson's AI-driven operations platform uses machine learning to predict network issues before they impact subscribers. Predictive maintenance algorithms can forecast equipment failures with 95% accuracy up to 7 days in advance.

## Energy Efficiency
Ericsson's energy-efficient 5G products reduce power consumption by up to 25% compared to previous generations. The company aims to achieve zero net carbon emissions by 2040.

## Private Networks
Ericsson Private 5G Networks provide dedicated connectivity for enterprise and industrial applications including smart manufacturing, port operations, and mining with autonomous vehicles.

## Edge Computing
Ericsson's edge computing solutions bring processing power closer to users, reducing latency to under 10 milliseconds. Essential for augmented reality and autonomous vehicles.

## Security Framework
Ericsson's 5G security framework implements zero-trust architecture with continuous verification of all network entities and 24/7 monitoring.
"""

# Create chunks
sections = re.split(r'\n##\s+', ERICSSON_KB)
section_chunks = [s.strip() for s in sections if s.strip() and len(s.strip()) > 50]
print(f"✅ Created {len(section_chunks)} chunks")

# Setup embedding client
embedding_client = AzureOpenAI(
    api_key=EMBEDDING_KEY,
    api_version="2024-02-01",
    azure_endpoint=EMBEDDING_ENDPOINT
)

def get_embedding(text):
    response = embedding_client.embeddings.create(input=text, model=EMBEDDING_DEPLOYMENT)
    return response.data[0].embedding

# Create vector store
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("ericsson_knowledge")
except:
    pass

collection = chroma_client.create_collection(name="ericsson_knowledge")

print("🔄 Creating embeddings...")
for i, chunk in enumerate(section_chunks):
    embedding = get_embedding(chunk)
    topic = chunk.split('\n')[0][:30]
    collection.add(ids=[f"chunk_{i}"], embeddings=[embedding], documents=[chunk], metadatas=[{"chunk_id": i}])
    print(f"   ✅ Chunk {i+1}: {topic}...")

print(f"\n✅ Vector database ready with {collection.count()} chunks!")

ModuleNotFoundError: No module named 'openai'

In [4]:
!pip install openai chromadb

In [1]:
import openai
print(f"✅ OpenAI version: {openai.__version__}")
import chromadb
print(f"✅ ChromaDB version: {chromadb.__version__}")

ModuleNotFoundError: No module named 'openai'

In [2]:
import openai
print(f"✅ OpenAI version: {openai.__version__}")
import chromadb
print(f"✅ ChromaDB version: {chromadb.__version__}")

ModuleNotFoundError: No module named 'openai'

In [3]:
✅ OpenAI version: 1.x.x
✅ ChromaDB version: 0.x.x

SyntaxError: invalid character '✅' (U+2705) (758862863.py, line 1)

In [4]:
✅ OpenAI version: 1.x.x
✅ ChromaDB version: 0.x.x

SyntaxError: invalid character '✅' (U+2705) (758862863.py, line 1)

In [5]:
import openai
print("OpenAI version:", openai.__version__)
import chromadb
print("ChromaDB version:", chromadb.__version__)

ModuleNotFoundError: No module named 'openai'

In [1]:
import sys
!{sys.executable} -m pip install openai chromadb --quiet
print("Done! Now run the next cell.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azure-monitor-opentelemetry 1.6.10 requires opentelemetry-sdk<1.32,>=1.28.0, but you have opentelemetry-sdk 1.39.0 which is incompatible.
mlflow 3.1.1 requires mlflow-skinny==3.1.1, but you have mlflow-skinny 2.22.1 which is incompatible.
mlflow-skinny 2.22.1 requires packaging<25, but you have packaging 25.0 which is incompatible.
opentelemetry-instrumentation 0.52b1 requires opentelemetry-semantic-conventions==0.52b1, but you have opentelemetry-semantic-conventions 0.60b0 which is incompatible.
opentelemetry-instrumentation-asgi 0.52b1 requires opentelemetry-semantic-conventions==0.52b1, but you have opentelemetry-semantic-conventions 0.60b0 which is incompatible.
opentelemetry-instrumentation-dbapi 0.52b1 requires opentelemetry-semantic-conventions==0.52b1, but you have opentelemetry-semantic-conventions 0.60b0

In [2]:
import openai
print("OpenAI version:", openai.__version__)

OpenAI version: 2.9.0


In [3]:
# Complete RAG Setup
from openai import AzureOpenAI
import chromadb
import re

# Configuration
EMBEDDING_ENDPOINT = "https://rag-demo-openai-najib.openai.azure.com/"
EMBEDDING_KEY = "22eglYDwbQGJhdtByt2SeLDNVS1i5PnAW0R3nqnq4AGX3ml3tNi3JQQJ99BLACYeBjFX"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"
GPT_ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"
GPT_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgx

SyntaxError: unterminated string literal (detected at line 11) (3725651660.py, line 11)

In [4]:
# COMPREHENSIVE RAG DEMO FOR ERICSSON
from openai import AzureOpenAI
import chromadb
import re

print("=" * 70)
print("        ERICSSON RAG DEMO - CHUNKING, EMBEDDING & RETRIEVAL")
print("=" * 70)

# Configuration
EMBEDDING_ENDPOINT = "https://rag-demo-openai-najib.openai.azure.com/"
EMBEDDING_KEY = "22eglYDwbQGJhdtByt2SeLDNVS1i5PnAW0R3nqnq4AGX3ml3tNi3JQQJ99BLACYeBjFX"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"
GPT_ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"
GPT_DEPLOYMENT = "gpt-4o-mini"

# GPT Key (split to avoid line issues)
GPT_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"

# Ericsson Knowledge Base
ERICSSON_KB = """
## Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases such as enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC).

## Ericsson Spectrum Sharing
Ericsson Spectrum Sharing (ESS) allows operators to dynamically share spectrum between 4G LTE and 5G NR on the same frequency band. This technology uses intelligent algorithms to allocate spectrum resources in real-time based on traffic demand.

## Cloud RAN Architecture
Ericsson's Cloud RAN solution virtualizes radio access network functions, enabling deployment on commercial off-the-shelf hardware. Cloud RAN reduces total cost of ownership by up to 40% through hardware consolidation.

## AI-Powered Network Operations
Ericsson's AI-driven operations platform uses machine learning to predict network issues before they impact subscribers. Predictive maintenance algorithms can forecast equipment failures with 95% accuracy up to 7 days in advance.

## Energy Efficiency
Ericsson's energy-efficient 5G products reduce power consumption by up to 25% compared to previous generations. The company aims to achieve zero net carbon emissions by 2040.

## Private Networks
Ericsson Private 5G Networks provide dedicated connectivity for enterprise and industrial applications including smart manufacturing, port operations, and mining with autonomous vehicles.

## Edge Computing
Ericsson's edge computing solutions bring processing power closer to users, reducing latency to under 10 milliseconds. Essential for augmented reality and autonomous vehicles.

## Security Framework
Ericsson's 5G security framework implements zero-trust architecture with continuous verification of all network entities and 24/7 monitoring.
"""

print("\n[STEP 1] KNOWLEDGE BASE LOADED")
print(f"Characters: {len(ERICSSON_KB)}, Words: {len(ERICSSON_KB.split())}")

# Chunking
print("\n" + "=" * 70)
print("[STEP 2] CHUNKING STRATEGIES")
print("=" * 70)

def fixed_size_chunking(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]

def section_chunking(text):
    sections = re.split(r'\n##\s+', text)
    return [s.strip() for s in sections if s.strip() and len(s.strip()) > 50]

fixed_chunks = fixed_size_chunking(ERICSSON_KB)
section_chunks = section_chunking(ERICSSON_KB)

print(f"\nFixed-Size (500 chars): {len(fixed_chunks)} chunks")
print(f"Section-Based (by topic): {len(section_chunks)} chunks")
print("\n--> Using Section-Based for best RAG results")

# Show chunks
print("\nOur semantic chunks:")
for i, chunk in enumerate(section_chunks):
    topic = chunk.split('\n')[0][:40]
    print(f"  Chunk {i+1}: {topic}... ({len(chunk)} chars)")

# Embeddings
print("\n" + "=" * 70)
print("[STEP 3] CREATING EMBEDDINGS")
print("=" * 70)

embedding_client = AzureOpenAI(
    api_key=EMBEDDING_KEY,
    api_version="2024-02-01",
    azure_endpoint=EMBEDDING_ENDPOINT
)

def get_embedding(text):
    response = embedding_client.embeddings.create(input=text, model=EMBEDDING_DEPLOYMENT)
    return response.data[0].embedding

test_emb = get_embedding("test")
print(f"Embedding dimension: {len(test_emb)}")

# Vector Store
print("\n" + "=" * 70)
print("[STEP 4] VECTOR DATABASE")
print("=" * 70)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("ericsson_knowledge")
except:
    pass

collection = chroma_client.create_collection(name="ericsson_knowledge")

print("Adding chunks to ChromaDB...")
for i, chunk in enumerate(section_chunks):
    embedding = get_embedding(chunk)
    topic = chunk.split('\n')[0][:30]
    collection.add(ids=[f"chunk_{i}"], embeddings=[embedding], documents=[chunk], metadatas=[{"chunk_id": i, "topic": topic}])
    print(f"  Added: {topic}...")

print(f"\nVector database ready with {collection.count()} chunks!")
print("\n" + "=" * 70)
print("SETUP COMPLETE!")
print("=" * 70)

        ERICSSON RAG DEMO - CHUNKING, EMBEDDING & RETRIEVAL

[STEP 1] KNOWLEDGE BASE LOADED
Characters: 1893, Words: 251

[STEP 2] CHUNKING STRATEGIES

Fixed-Size (500 chars): 5 chunks
Section-Based (by topic): 8 chunks

--> Using Section-Based for best RAG results

Our semantic chunks:
  Chunk 1: Network Slicing... (337 chars)
  Chunk 2: Ericsson Spectrum Sharing... (270 chars)
  Chunk 3: Cloud RAN Architecture... (241 chars)
  Chunk 4: AI-Powered Network Operations... (259 chars)
  Chunk 5: Energy Efficiency... (192 chars)
  Chunk 6: Private Networks... (204 chars)
  Chunk 7: Edge Computing... (190 chars)
  Chunk 8: Security Framework... (160 chars)

[STEP 3] CREATING EMBEDDINGS


AuthenticationError: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}

In [5]:
# COMPREHENSIVE RAG DEMO FOR ERICSSON
from openai import AzureOpenAI
import chromadb
import re

print("=" * 70)
print("        ERICSSON RAG DEMO")
print("=" * 70)

# CORRECTED Configuration
EMBEDDING_ENDPOINT = "https://rag-demo-openai-najib.openai.azure.com/"
EMBEDDING_KEY = "22eglYDwbQGJhdtByt2SeLDNVS1i5PnAW0R3nqnq4AGX3ml3tNi3JQQJ99BLACYeBjFXJ3w3AAABACOGjPvN"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"
GPT_ENDPOINT = "https://najr-miyonro1-eastus2.cognitiveservices.azure.com/"
GPT_KEY = "9sCteGVmfDgbLsSu5V4znBU3bmB08BfjmisPWAuJEHRx0njOZdSMJQQJ99BLACHYHv6XJ3w3AAAAACOGgxKS"
GPT_DEPLOYMENT = "gpt-4o-mini"

# Ericsson Knowledge Base
ERICSSON_KB = """
## Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases such as enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC).

## Ericsson Spectrum Sharing
Ericsson Spectrum Sharing (ESS) allows operators to dynamically share spectrum between 4G LTE and 5G NR on the same frequency band. This technology uses intelligent algorithms to allocate spectrum resources in real-time.

## Cloud RAN Architecture
Ericsson's Cloud RAN solution virtualizes radio access network functions, enabling deployment on commercial off-the-shelf hardware. Cloud RAN reduces total cost of ownership by up to 40%.

## AI-Powered Network Operations
Ericsson's AI-driven operations platform uses machine learning to predict network issues before they impact subscribers. Predictive maintenance can forecast equipment failures with 95% accuracy.

## Energy Efficiency
Ericsson's energy-efficient 5G products reduce power consumption by up to 25%. The company aims for zero net carbon emissions by 2040.

## Private Networks
Ericsson Private 5G Networks provide dedicated connectivity for enterprise applications including smart manufacturing and mining operations.

## Edge Computing
Ericsson's edge computing reduces latency to under 10 milliseconds. Essential for augmented reality and autonomous vehicles.

## Security Framework
Ericsson's 5G security implements zero-trust architecture with 24/7 monitoring.
"""

# Create section chunks
section_chunks = [s.strip() for s in re.split(r'\n##\s+', ERICSSON_KB) if s.strip() and len(s.strip()) > 50]
print(f"\n[STEP 1] Created {len(section_chunks)} chunks from knowledge base")

# Setup embedding client
embedding_client = AzureOpenAI(
    api_key=EMBEDDING_KEY,
    api_version="2024-02-01",
    azure_endpoint=EMBEDDING_ENDPOINT
)

def get_embedding(text):
    response = embedding_client.embeddings.create(input=text, model=EMBEDDING_DEPLOYMENT)
    return response.data[0].embedding

print("\n[STEP 2] Testing embedding API...")
test_emb = get_embedding("test")
print(f"Embedding works! Dimension: {len(test_emb)}")

# Vector Store
print("\n[STEP 3] Creating vector database...")
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("ericsson_knowledge")
except:
    pass

collection = chroma_client.create_collection(name="ericsson_knowledge")

for i, chunk in enumerate(section_chunks):
    embedding = get_embedding(chunk)
    topic = chunk.split('\n')[0][:30]
    collection.add(ids=[f"chunk_{i}"], embeddings=[embedding], documents=[chunk], metadatas=[{"chunk_id": i, "topic": topic}])
    print(f"  Added: {topic}...")

print(f"\n[STEP 4] Vector database ready with {collection.count()} chunks!")
print("\n" + "=" * 70)
print("SETUP COMPLETE!")
print("=" * 70)

        ERICSSON RAG DEMO

[STEP 1] Created 8 chunks from knowledge base

[STEP 2] Testing embedding API...
Embedding works! Dimension: 1536

[STEP 3] Creating vector database...
  Added: Network Slicing...
  Added: Ericsson Spectrum Sharing...
  Added: Cloud RAN Architecture...
  Added: AI-Powered Network Operations...
  Added: Energy Efficiency...
  Added: Private Networks...
  Added: Edge Computing...
  Added: Security Framework...

[STEP 4] Vector database ready with 8 chunks!

SETUP COMPLETE!


In [6]:
# RETRIEVAL DEMO - Search by meaning!
print("=" * 70)
print("        RETRIEVAL DEMO - SEMANTIC SEARCH")
print("=" * 70)

def search_knowledge(query, n_results=3):
    """Search the vector database for relevant chunks"""
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results

# Test queries
test_queries = [
    "How can I reduce network latency?",
    "What is Ericsson doing for sustainability?",
    "How does 5G handle different use cases?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print("="*60)
    
    results = search_knowledge(query, n_results=2)
    
    for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
        similarity = 1 - distance  # Convert distance to similarity
        topic = doc.split('\n')[0][:40]
        print(f"\n  Result {i+1} (similarity: {similarity:.2f}):")
        print(f"  Topic: {topic}")
        print(f"  Content: {doc[:150]}...")

print("\n" + "=" * 70)
print("Notice: The search finds RELEVANT content by MEANING, not keywords!")
print("=" * 70)

        RETRIEVAL DEMO - SEMANTIC SEARCH

QUERY: How can I reduce network latency?

  Result 1 (similarity: 0.63):
  Topic: Edge Computing
  Content: Edge Computing
Ericsson's edge computing reduces latency to under 10 milliseconds. Essential for augmented reality and autonomous vehicles....

  Result 2 (similarity: 0.57):
  Topic: Network Slicing
  Content: Network Slicing
Ericsson's network slicing technology enables operators to create multiple virtual networks on a single physical infrastructure. Each ...

QUERY: What is Ericsson doing for sustainability?

  Result 1 (similarity: 0.75):
  Topic: Energy Efficiency
  Content: Energy Efficiency
Ericsson's energy-efficient 5G products reduce power consumption by up to 25%. The company aims for zero net carbon emissions by 204...

  Result 2 (similarity: 0.66):
  Topic: Edge Computing
  Content: Edge Computing
Ericsson's edge computing reduces latency to under 10 milliseconds. Essential for augmented reality and autonomous vehicles....


In [7]:
# COMPLETE RAG DEMO - Question Answering
print("=" * 70)
print("        COMPLETE RAG PIPELINE - ASK QUESTIONS!")
print("=" * 70)

# Setup GPT client
gpt_client = AzureOpenAI(
    api_key=GPT_KEY,
    api_version="2024-02-01",
    azure_endpoint=GPT_ENDPOINT
)

def rag_query(question, n_chunks=3):
    """Complete RAG pipeline: Retrieve + Generate"""
    
    print(f"\n[1] QUESTION: {question}")
    print("-" * 50)
    
    # Step 1: Retrieve relevant chunks
    print("\n[2] RETRIEVING relevant chunks...")
    query_embedding = get_embedding(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_chunks
    )
    
    retrieved_chunks = results['documents'][0]
    for i, chunk in enumerate(retrieved_chunks):
        topic = chunk.split('\n')[0][:35]
        print(f"    Retrieved: {topic}...")
    
    # Step 2: Build context from retrieved chunks
    context = "\n\n".join(retrieved_chunks)
    
    # Step 3: Generate answer using GPT
    print("\n[3] GENERATING answer with GPT...")
    
    prompt = f"""Based on the following context from Ericsson documentation, answer the question.
If the answer is not in the context, say "I don't have information about that."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = gpt_client.chat.completions.create(
        model=GPT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a helpful Ericsson technical assistant. Answer questions based only on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=300
    )
    
    answer = response.choices[0].message.content
    
    print("\n[4] ANSWER:")
    print("=" * 50)
    print(answer)
    print("=" * 50)
    
    return answer

# Demo questions
print("\n" + "#" * 70)
print("# DEMO: Asking questions about Ericsson products")
print("#" * 70)

# Question 1
rag_query("What is network slicing and what are its use cases?")

# Question 2
rag_query("How is Ericsson addressing energy efficiency and sustainability?")

# Question 3
rag_query("What AI capabilities does Ericsson offer for network operations?")

        COMPLETE RAG PIPELINE - ASK QUESTIONS!

######################################################################
# DEMO: Asking questions about Ericsson products
######################################################################

[1] QUESTION: What is network slicing and what are its use cases?
--------------------------------------------------

[2] RETRIEVING relevant chunks...
    Retrieved: Network Slicing...
    Retrieved: Private Networks...
    Retrieved: Ericsson Spectrum Sharing...

[3] GENERATING answer with GPT...

[4] ANSWER:
Network slicing is Ericsson's technology that enables operators to create multiple virtual networks on a single physical infrastructure. Each slice can be customized for specific use cases, including enhanced mobile broadband (eMBB), ultra-reliable low-latency communications (URLLC), and massive machine-type communications (mMTC).

[1] QUESTION: How is Ericsson addressing energy efficiency and sustainability?
----------------------------------

'Ericsson offers AI capabilities that include a machine learning-driven operations platform that predicts network issues before they impact subscribers. Additionally, it provides predictive maintenance that can forecast equipment failures with 95% accuracy.'

In [8]:
# INTERACTIVE RAG - Ask your own questions!
print("=" * 70)
print("        INTERACTIVE RAG DEMO")
print("=" * 70)

# You can change this question to anything!
your_question = "What solutions does Ericsson offer for reducing latency?"

print(f"\nYour Question: {your_question}\n")
rag_query(your_question)

# Try these other questions by changing the variable above:
# - "How does Cloud RAN reduce costs?"
# - "What security features does Ericsson 5G have?"
# - "How can operators deploy 5G faster?"
# - "What is spectrum sharing?"

        INTERACTIVE RAG DEMO

Your Question: What solutions does Ericsson offer for reducing latency?


[1] QUESTION: What solutions does Ericsson offer for reducing latency?
--------------------------------------------------

[2] RETRIEVING relevant chunks...
    Retrieved: Edge Computing...
    Retrieved: Energy Efficiency...
    Retrieved: Network Slicing...

[3] GENERATING answer with GPT...

[4] ANSWER:
Ericsson offers edge computing solutions that reduce latency to under 10 milliseconds, which is essential for applications like augmented reality and autonomous vehicles.


'Ericsson offers edge computing solutions that reduce latency to under 10 milliseconds, which is essential for applications like augmented reality and autonomous vehicles.'

In [9]:
# SUMMARY FOR YOUR TEAM PRESENTATION
print("""
================================================================================
          RAG IMPLEMENTATION SUMMARY - FOR ERICSSON TEAM
================================================================================

WHAT WE BUILT TODAY:
--------------------
A complete Retrieval-Augmented Generation (RAG) system using Azure OpenAI
that can answer questions about Ericsson products from internal documentation.

ARCHITECTURE:
-------------
    [Ericsson Docs] --> [Chunking] --> [Embedding] --> [Vector DB]
                                                            |
    [User Question] --> [Embed Query] --> [Similarity Search]
                                                            |
                                         [Retrieved Chunks] + [Question]
                                                            |
                                                   [GPT generates answer]
                                                            |
                                                    [Grounded Answer]

COMPONENTS USED:
----------------
1. Azure OpenAI (text-embedding-ada-002) - Creates 1536-dimension vectors
2. Azure OpenAI (gpt-4o-mini) - Generates answers
3. ChromaDB - Vector database for storing/searching embeddings
4. Python - Orchestration

KEY CONCEPTS DEMONSTRATED:
--------------------------
1. CHUNKING STRATEGIES:
   - Fixed-size: Simple but cuts mid-sentence (bad for RAG)
   - Sentence-based: Preserves grammar but may split topics
   - Section-based: Keeps topics together (best for RAG)

2. EMBEDDINGS:
   - Convert text to vectors (numbers)
   - Similar meanings = similar vectors
   - Enables semantic search (search by meaning, not keywords)

3. RETRIEVAL:
   - Query is converted to vector
   - Find most similar chunks using cosine similarity
   - Return top-K relevant chunks

4. GENERATION:
   - Retrieved chunks become "context"
   - GPT generates answer using ONLY the context
   - Result is grounded in your data (no hallucination)

WHY RAG MATTERS FOR ERICSSON:
-----------------------------
- LLMs don't know internal Ericsson documentation
- RAG lets us query internal knowledge bases
- Answers are grounded in YOUR data
- Can be applied to: Product docs, Support tickets, Technical specs, etc.

NEXT STEPS FOR PRODUCTION:
--------------------------
1. Replace ChromaDB with: Pinecone, Weaviate, or Azure AI Search
2. Add more documents to knowledge base
3. Implement document ingestion pipeline
4. Add evaluation metrics (relevance, accuracy)
5. Deploy as API or chatbot

AZURE RESOURCES CREATED:
------------------------
- Azure OpenAI (rag-demo-openai-najib) - East US
- Azure OpenAI (najr-miyonro1-eastus2) - East US 2
- Azure ML Workspace (rag-demo-ml-workspace)
- Compute Instance (rag-demo-compute-najib)

================================================================================
                    DEMO COMPLETE - QUESTIONS?
================================================================================
""")


          RAG IMPLEMENTATION SUMMARY - FOR ERICSSON TEAM

WHAT WE BUILT TODAY:
--------------------
A complete Retrieval-Augmented Generation (RAG) system using Azure OpenAI
that can answer questions about Ericsson products from internal documentation.

ARCHITECTURE:
-------------
    [Ericsson Docs] --> [Chunking] --> [Embedding] --> [Vector DB]
                                                            |
    [User Question] --> [Embed Query] --> [Similarity Search]
                                                            |
                                         [Retrieved Chunks] + [Question]
                                                            |
                                                   [GPT generates answer]
                                                            |
                                                    [Grounded Answer]

COMPONENTS USED:
----------------
1. Azure OpenAI (text-embedding-ada-002) - Creates 1536-dimension vectors
2. Azure OpenA

In [10]:
# ADVANCED RAG CONCEPTS - COMPLETE GUIDE
print("=" * 70)
print("        ADVANCED RAG CONCEPTS FOR PRODUCTION")
print("=" * 70)

# ============================================================
# 1. RETRIEVAL TYPES
# ============================================================
print("""
[1] RETRIEVAL TYPES
================================================================================

There are several ways to retrieve relevant documents:

+------------------+----------------------------------------------------------+
| RETRIEVAL TYPE   | DESCRIPTION                                              |
+------------------+----------------------------------------------------------+
| SIMILARITY       | Returns chunks most similar to query (what we used)      |
| SEARCH           | - Uses cosine similarity                                 |
|                  | - Fast and simple                                        |
|                  | - May return redundant results                           |
+------------------+----------------------------------------------------------+
| MMR (Maximal     | Balances relevance AND diversity                         |
| Marginal         | - Avoids returning similar chunks                        |
| Relevance)       | - Better coverage of different aspects                   |
|                  | - Use when you want varied information                   |
+------------------+----------------------------------------------------------+
| HYBRID SEARCH    | Combines keyword search + semantic search                |
|                  | - Best of both worlds                                    |
|                  | - BM25 (keywords) + Vector similarity                    |
|                  | - Useful when exact terms matter (product codes, etc.)   |
+------------------+----------------------------------------------------------+
| SELF-QUERY       | LLM extracts filters from natural language               |
|                  | - "Show me 5G docs from 2024"                            |
|                  | - Automatically filters by metadata                      |
+------------------+----------------------------------------------------------+
| PARENT DOCUMENT  | Retrieves small chunks, returns larger parent            |
|                  | - Search on sentences, return full sections              |
|                  | - Better context for generation                          |
+------------------+----------------------------------------------------------+
| MULTI-QUERY      | Generates multiple query variations                      |
|                  | - Improves recall                                        |
|                  | - "network latency" -> also searches "delay", "lag"      |
+------------------+----------------------------------------------------------+

For Ericsson: Start with SIMILARITY, upgrade to HYBRID for production.
""")

# ============================================================
# 2. RETRIEVAL DEMO WITH DIFFERENT TYPES
# ============================================================
print("\n" + "=" * 70)
print("[2] RETRIEVAL COMPARISON DEMO")
print("=" * 70)

query = "How does Ericsson handle network performance?"

# Standard Similarity Search
print(f"\nQuery: '{query}'")
print("\n--- SIMILARITY SEARCH (top 3) ---")
results = collection.query(
    query_embeddings=[get_embedding(query)],
    n_results=3
)
for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
    topic = doc.split('\n')[0][:40]
    similarity = 1 - dist
    print(f"  {i+1}. [{similarity:.2f}] {topic}")

# MMR-style (manual implementation for demo)
print("\n--- MMR-STYLE (diverse results) ---")
print("  MMR would avoid returning similar chunks")
print("  Example: If 'AI Operations' and 'Cloud RAN' are both about performance,")
print("  MMR picks one and finds diverse alternatives")

# ============================================================
# 3. CITATION AND SOURCE TRACKING
# ============================================================
print("\n" + "=" * 70)
print("[3] CITATION AND SOURCE TRACKING")
print("=" * 70)

print("""
WHY CITATIONS MATTER:
- Users need to verify information
- Builds trust in the system
- Required for enterprise/compliance use cases
- Helps debug retrieval issues

HOW TO IMPLEMENT CITATIONS:
""")

def rag_with_citations(question, n_chunks=3):
    """RAG with source citations"""
    
    # Retrieve with metadata
    query_embedding = get_embedding(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_chunks,
        include=["documents", "metadatas", "distances"]
    )
    
    # Build context with source markers
    context_parts = []
    sources = []
    
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0], 
        results['metadatas'][0],
        results['distances'][0]
    )):
        source_id = f"[Source {i+1}]"
        topic = doc.split('\n')[0]
        sources.append({
            "id": source_id,
            "topic": topic,
            "similarity": round(1 - dist, 2),
            "chunk_id": meta.get('chunk_id', i)
        })
        context_parts.append(f"{source_id}\n{doc}")
    
    context = "\n\n".join(context_parts)
    
    # Prompt that encourages citations
    prompt = f"""Answer the question using ONLY the provided sources.
Include [Source X] citations for each fact you mention.

SOURCES:
{context}

QUESTION: {question}

ANSWER (with citations):"""

    response = gpt_client.chat.completions.create(
        model=GPT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Always cite your sources using [Source X] format."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=400
    )
    
    answer = response.choices[0].message.content
    
    return answer, sources

# Demo with citations
print("\nDEMO: Question with Citations")
print("-" * 50)
question = "What AI and automation capabilities does Ericsson offer?"
answer, sources = rag_with_citations(question)

print(f"Question: {question}\n")
print(f"Answer:\n{answer}\n")
print("Sources Used:")
for src in sources:
    print(f"  {src['id']}: {src['topic'][:40]}... (similarity: {src['similarity']})")

# ============================================================
# 4. HALLUCINATION PREVENTION
# ============================================================
print("\n" + "=" * 70)
print("[4] HALLUCINATION PREVENTION TECHNIQUES")
print("=" * 70)

print("""
WHAT IS HALLUCINATION?
- LLM generates plausible but FALSE information
- Not grounded in provided context
- Dangerous for enterprise use cases

PREVENTION TECHNIQUES:

+---------------------------+------------------------------------------------+
| TECHNIQUE                 | HOW IT WORKS                                   |
+---------------------------+------------------------------------------------+
| 1. STRICT PROMPTING       | "Answer ONLY from the context provided"        |
|                           | "If not in context, say 'I don't know'"        |
+---------------------------+------------------------------------------------+
| 2. LOW TEMPERATURE        | temperature=0.1 to 0.3 (less creative)         |
|                           | Higher = more hallucination risk               |
+---------------------------+------------------------------------------------+
| 3. CITATION REQUIREMENT   | Force model to cite sources                    |
|                           | If it can't cite, it shouldn't say it          |
+---------------------------+------------------------------------------------+
| 4. RETRIEVAL THRESHOLD    | Only use chunks with similarity > 0.7          |
|                           | Low similarity = probably not relevant         |
+---------------------------+------------------------------------------------+
| 5. CONFIDENCE SCORING     | Ask model: "How confident are you? (1-10)"     |
|                           | Filter low-confidence answers                  |
+---------------------------+------------------------------------------------+
| 6. FACT VERIFICATION      | Second LLM call to verify against sources      |
|                           | "Is this answer supported by the context?"     |
+---------------------------+------------------------------------------------+
| 7. ABSTAIN OPTION         | Train/prompt model to say "I don't know"       |
|                           | Better than wrong answer                       |
+---------------------------+------------------------------------------------+

OUR IMPLEMENTATION:
- We use temperature=0.2 (low creativity)
- Prompt says "Answer ONLY from context"
- Prompt says "If not in context, say I don't have information"
""")

# Demo: Testing hallucination prevention
print("\nDEMO: Hallucination Prevention Test")
print("-" * 50)
out_of_scope_question = "What is Ericsson's stock price today?"
print(f"Question (out of scope): {out_of_scope_question}")
answer, _ = rag_with_citations(out_of_scope_question)
print(f"Answer: {answer}")
print("\n--> Good RAG should say 'I don't have information about that'")

# ============================================================
# 5. PROMPT TEMPLATES
# ============================================================
print("\n" + "=" * 70)
print("[5] PROMPT TEMPLATES FOR RAG")
print("=" * 70)

print("""
Different prompt templates for different use cases:

TEMPLATE 1: BASIC QA
--------------------
Context: {context}
Question: {question}
Answer:

TEMPLATE 2: WITH CITATIONS (what we use)
----------------------------------------
Answer the question using ONLY the provided sources.
Include [Source X] citations for each fact.

Sources:
{context}

Question: {question}
Answer (with citations):

TEMPLATE 3: STRICT NO-HALLUCINATION
-----------------------------------
You are a precise assistant. Answer based ONLY on the given context.
- If the answer is in the context, provide it with citations
- If the answer is NOT in the context, respond: "I cannot find this information in the available documents."
- Never make up information

Context:
{context}

Question: {question}
Answer:

TEMPLATE 4: CONVERSATIONAL
--------------------------
You are a helpful Ericsson product expert. Use the following documentation 
to answer the customer's question in a friendly, professional tone.

Documentation:
{context}

Customer Question: {question}
Your Response:

TEMPLATE 5: TECHNICAL EXPERT
----------------------------
You are a senior Ericsson network architect. Provide a detailed technical 
response based on the following documentation. Include specific metrics 
and technical details where available.

Technical Documentation:
{context}

Technical Query: {question}
Expert Analysis:

TEMPLATE 6: SUMMARIZATION
-------------------------
Summarize the key points from the following Ericsson documentation 
that are relevant to the question.

Documentation:
{context}

Topic: {question}
Summary:
""")

# ============================================================
# 6. EVALUATION METRICS
# ============================================================
print("\n" + "=" * 70)
print("[6] RAG EVALUATION METRICS")
print("=" * 70)

print("""
HOW TO MEASURE RAG QUALITY:

+-------------------+----------------------------------------------------------+
| METRIC            | WHAT IT MEASURES                                         |
+-------------------+----------------------------------------------------------+
| RETRIEVAL METRICS |                                                          |
+-------------------+----------------------------------------------------------+
| Precision@K       | % of retrieved docs that are relevant                    |
|                   | High = few irrelevant results                            |
+-------------------+----------------------------------------------------------+
| Recall@K          | % of relevant docs that were retrieved                   |
|                   | High = didn't miss important info                        |
+-------------------+----------------------------------------------------------+
| MRR (Mean         | Where does first relevant result appear?                 |
| Reciprocal Rank)  | Higher = relevant docs ranked higher                     |
+-------------------+----------------------------------------------------------+
| NDCG              | Quality of ranking (position matters)                    |
+-------------------+----------------------------------------------------------+
| GENERATION METRICS|                                                          |
+-------------------+----------------------------------------------------------+
| Faithfulness      | Is answer supported by retrieved context?                |
|                   | (No hallucination)                                       |
+-------------------+----------------------------------------------------------+
| Answer Relevance  | Does answer address the question?                        |
+-------------------+----------------------------------------------------------+
| Context Relevance | Were retrieved chunks actually relevant?                 |
+-------------------+----------------------------------------------------------+
| Groundedness      | Every claim traceable to source?                         |
+-------------------+----------------------------------------------------------+

EVALUATION TOOLS:
- RAGAS (Python library) - Automatic RAG evaluation
- LangSmith - Tracing and evaluation
- TruLens - Feedback functions
- Human evaluation - Gold standard but expensive

SIMPLE EVALUATION APPROACH:
1. Create test set: 20-50 question-answer pairs
2. Run RAG on questions
3. Compare to expected answers
4. Calculate accuracy, faithfulness
""")

print("\n" + "=" * 70)
print("        ADVANCED CONCEPTS COMPLETE!")
print("=" * 70)


        ADVANCED RAG CONCEPTS FOR PRODUCTION

[1] RETRIEVAL TYPES

There are several ways to retrieve relevant documents:

+------------------+----------------------------------------------------------+
| RETRIEVAL TYPE   | DESCRIPTION                                              |
+------------------+----------------------------------------------------------+
| SIMILARITY       | Returns chunks most similar to query (what we used)      |
| SEARCH           | - Uses cosine similarity                                 |
|                  | - Fast and simple                                        |
|                  | - May return redundant results                           |
+------------------+----------------------------------------------------------+
| MMR (Maximal     | Balances relevance AND diversity                         |
| Marginal         | - Avoids returning similar chunks                        |
| Relevance)       | - Better coverage of different aspects                  

In [11]:
# HOW TO EXPORT JUPYTER NOTEBOOK
print("""
================================================================================
        HOW TO EXPORT YOUR JUPYTER NOTEBOOK
================================================================================

METHOD 1: DOWNLOAD AS NOTEBOOK (.ipynb)
---------------------------------------
1. Click "File" menu at top (or the three dots ...)
2. Select "Download" or "Export"
3. Choose "Download as .ipynb"
4. This saves the complete notebook with all code and outputs

METHOD 2: DOWNLOAD AS HTML (Best for sharing!)
----------------------------------------------
1. File -> Download as -> HTML
2. Creates a single HTML file with all outputs visible
3. Anyone can open in browser - no Python needed!
4. Perfect for sharing with team

METHOD 3: DOWNLOAD AS PDF
-------------------------
1. File -> Download as -> PDF
2. May require additional setup
3. Alternative: Download as HTML, then print to PDF from browser

METHOD 4: EXPORT VIA COMMAND LINE
---------------------------------
Run these commands in a notebook cell:

# Export to HTML
!jupyter nbconvert --to html rag_demo_ericsson.ipynb

# Export to PDF (requires latex)
!jupyter nbconvert --to pdf rag_demo_ericsson.ipynb

# Export to Python script
!jupyter nbconvert --to script rag_demo_ericsson.ipynb

METHOD 5: COPY ALL OUTPUTS
--------------------------
1. Click on a cell output
2. Ctrl+A (select all)
3. Ctrl+C (copy)
4. Paste into Word/Google Docs

================================================================================
        FOR AZURE ML NOTEBOOKS SPECIFICALLY:
================================================================================

In Azure ML Studio:
1. Right-click on your notebook file in the left panel
2. Select "Download"
3. This downloads the .ipynb file

To get HTML export in Azure ML:
Run this cell below to convert and download.
""")


        HOW TO EXPORT YOUR JUPYTER NOTEBOOK

METHOD 1: DOWNLOAD AS NOTEBOOK (.ipynb)
---------------------------------------
1. Click "File" menu at top (or the three dots ...)
2. Select "Download" or "Export"
3. Choose "Download as .ipynb"
4. This saves the complete notebook with all code and outputs

METHOD 2: DOWNLOAD AS HTML (Best for sharing!)
----------------------------------------------
1. File -> Download as -> HTML
2. Creates a single HTML file with all outputs visible
3. Anyone can open in browser - no Python needed!
4. Perfect for sharing with team

METHOD 3: DOWNLOAD AS PDF
-------------------------
1. File -> Download as -> PDF
2. May require additional setup
3. Alternative: Download as HTML, then print to PDF from browser

METHOD 4: EXPORT VIA COMMAND LINE
---------------------------------
Run these commands in a notebook cell:

# Export to HTML
!jupyter nbconvert --to html rag_demo_ericsson.ipynb

# Export to PDF (requires latex)
!jupyter nbconvert --to pdf rag_demo_er